In [6]:
from sklearn.model_selection import TimeSeriesSplit
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from tqdm import tqdm

    item_id  year  month  seq   hs4    weight  quantity     value
0  DEWLVASR  2022      1  1.0  3038   14858.0       0.0   32688.0
1  ELQGMQWE  2022      1  1.0  2002   62195.0       0.0  110617.0
2  AHMDUILJ  2022      1  1.0  2102   18426.0       0.0   72766.0
3  XIPPENFQ  2022      1  1.0  2501   20426.0       0.0   11172.0
4  FTSVTTSR  2022      1  1.0  2529  248000.0       0.0  143004.0


설정값:
- 상관계수 임계값: 0.4


설정값:
- 상관계수 임계값: 0.4


공행성 쌍 탐색: 100it [00:08, 11.74it/s]

설정값:
- 상관계수 임계값: 0.4


공행성 쌍 탐색: 100it [00:08, 11.74it/s]


📊 검정 결과:
- 총 발견된 공행성 쌍: 1453

탐색된 공행성쌍 수: 1453


In [3]:
def plot_cv_indices(cv, X_length, ax=None):
    """TimeSeriesSplit의 분할을 시각화하는 함수"""
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 8))
    
    n_splits = cv.get_n_splits()
    
    for ii, (tr, tt) in enumerate(cv.split(range(X_length))):
        # Training set
        ax.fill_between(tr, [ii] * len(tr), [ii + 0.4] * len(tr), 
                       alpha=0.6, color='blue', label='Training' if ii == 0 else "")
        
        # Test set
        ax.fill_between(tt, [ii] * len(tt), [ii + 0.4] * len(tt), 
                       alpha=0.6, color='red', label='Test' if ii == 0 else "")
    
    ax.set_ylabel("CV Fold")
    ax.set_xlabel("Time Index")
    ax.set_title(f"TimeSeriesSplit Cross-Validation ({n_splits} folds)")
    ax.legend(loc='upper right')
    ax.set_ylim(-0.5, n_splits - 0.5)
    
    return ax

def prepare_time_series_data(pviot):
    """
    피벗 데이터를 시계열 분석을 위해 준비
    
    Returns:
    - time_index: 시간 인덱스 (월별 데이터)
    - item_data: 각 품목별 시계열 데이터
    """
    # 날짜 컬럼을 시간 순으로 정렬
    time_columns = sorted(pviot.columns)
    pivot_sorted = pviot[time_columns]
    
    print(f"📅 시계열 데이터 기간:")
    print(f"  시작: {time_columns[0]}")
    print(f"  종료: {time_columns[-1]}")
    print(f"  총 기간: {len(time_columns)}개월")
    
    return time_columns, pivot_sorted

# 시계열 데이터 준비
time_index, pivot_sorted = prepare_time_series_data(feature)
print(f"\n✅ 시계열 데이터 준비 완료!")
print(f"   시간축: {len(time_index)}개 시점")
print(f"   품목수: {len(pivot_sorted)}개")

📅 시계열 데이터 기간:
  시작: 2022-01-01 00:00:00
  종료: 2025-06-01 00:00:00
  총 기간: 42개월

✅ 시계열 데이터 준비 완료!
   시간축: 42개 시점
   품목수: 100개


In [8]:
from sklearn.model_selection import TimeSeriesSplit

# TimeSeriesSplit을 사용한 시계열 교차검증 설정 및 시각화
print("🚀 TimeSeriesSplit 설정 중...")

# TimeSeriesSplit 설정 (5개 fold)
tscv = TimeSeriesSplit(n_splits=5)
n_timepoints = len(time_index)

print(f"📊 교차검증 설정:")
print(f"  총 시점 수: {n_timepoints}")
print(f"  분할 수: {tscv.get_n_splits()}")
print(f"  최소 훈련 크기: {n_timepoints // (tscv.get_n_splits() + 1)}")


🚀 TimeSeriesSplit 설정 중...
📊 교차검증 설정:
  총 시점 수: 42
  분할 수: 5
  최소 훈련 크기: 7


In [60]:
def create_time_series_datasets(pivot_data, time_index, tscv):
    """
    TimeSeriesSplit를 사용하여 훈련/테스트 데이터셋 생성
    
    Returns:
    - datasets: [(X_train, X_test, y_train, y_test, train_dates, test_dates), ...]
    """
    datasets = []
    n_timepoints = len(time_index)
    
    print("📦 시계열 데이터셋 생성 중...")
    
    for fold, (train_idx, test_idx) in enumerate(tscv.split(range(n_timepoints))):
        # 날짜 정보
        train_dates = [time_index[i] for i in train_idx]
        test_dates = [time_index[i] for i in test_idx]
        
        # 데이터 분할
        X_train = pivot_data.iloc[:, train_idx[:-1]]  # 마지막 시점 제외 (target이 되기 때문)
        y_train = pivot_data.iloc[:, train_idx[-1]]   # 마지막 시점이 target
        
        X_test = pivot_data.iloc[:, test_idx[:-1]] if len(test_idx) > 1 else None
        y_test = pivot_data.iloc[:, test_idx[-1]]  # 테스트 target
        
        datasets.append({
            'fold': fold + 1,
            'X_train': X_train,
            'X_test': X_test,
            'y_train': y_train,
            'y_test': y_test,
            'train_dates': train_dates,
            'test_dates': test_dates,
            'train_idx': train_idx,
            'test_idx': test_idx
        })
        
        print(f"  Fold {fold + 1}: 훈련({len(train_dates)}개월) → 테스트({len(test_dates)}개월)")
    return datasets

time_series_datasets = create_time_series_datasets(pivot_sorted, time_index, tscv)

print(f"\n✅ {len(time_series_datasets)}개의 시계열 fold 데이터셋 생성 완료!")

📦 시계열 데이터셋 생성 중...
  Fold 1: 훈련(7개월) → 테스트(7개월)
  Fold 2: 훈련(14개월) → 테스트(7개월)
  Fold 3: 훈련(21개월) → 테스트(7개월)
  Fold 4: 훈련(28개월) → 테스트(7개월)
  Fold 5: 훈련(35개월) → 테스트(7개월)

✅ 5개의 시계열 fold 데이터셋 생성 완료!
